# 🪰 FlyFlappyBird: Embodied Drosophila MaleCNS Connectome Simulation

**A biophysically grounded closed-loop neural simulation where the complete *Drosophila* male central nervous system connectome (165K neurons, 6.2M synapses from Berg et al. 2026, *Cell*) plays Flappy Bird.**

This notebook runs seamlessly on both **Kaggle** (GPU T4/P100 or CPU) and **Local machines** (Apple Silicon M-Series MPS or CPU).

## 1. Environment & Project Root Setup
Automatically locates or clones the repository files so this notebook runs out-of-the-box in any environment (Kaggle, Colab, or local).

In [ ]:
import os
import sys
import subprocess

# ── 1. Locate or Clone Repository ─────────────────────────────────────────────
REPO_URL = "https://github.com/here-2007/Flappy-Bird-By-Male-CNS.git"
REPO_NAME = "Flappy-Bird-By-Male-CNS"

candidates = [
    os.getcwd(),
    os.path.join(os.getcwd(), REPO_NAME),
    os.path.abspath(os.path.join(os.getcwd(), "..")),
    os.path.join("/kaggle/working", REPO_NAME),
    "/kaggle/working",
]

found_root = None
for path in candidates:
    if os.path.exists(os.path.join(path, "config.py")):
        found_root = path
        break

if not found_root:
    print(f"[setup] Project files not found locally. Cloning {REPO_NAME} from GitHub...")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
    if os.path.exists(REPO_NAME):
        found_root = os.path.abspath(REPO_NAME)
    elif os.path.exists("config.py"):
        found_root = os.getcwd()

if found_root:
    os.chdir(found_root)
    if found_root not in sys.path:
        sys.path.insert(0, found_root)
    print(f"✅ Active Project Root: {os.getcwd()}")
else:
    raise RuntimeError("Could not find or clone project root containing config.py!")

## 2. Dependency Installation
Installs the required biophysics and graph dependencies.

In [ ]:
# Install required dependencies
!pip install -q -r requirements.txt

## 3. Hardware Acceleration & Dataset Detection
Inspect the compute device (NVIDIA CUDA T4/P100, Apple MPS, or CPU) and auto-detect Kaggle input dataset paths (`/kaggle/input/datasets/pernavjain/male-fruit-fly-cns/`).

In [ ]:
import torch
import config as CFG

device = CFG.get_device(verbose=True)
print(f"Device selected for sparse SpMV: {device}")
if device.type == "cuda":
    print(f"CUDA Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("Tip: If running on Kaggle, enable GPU acceleration: Session options -> Accelerator -> GPU T4 x2.")

print("\n--- Connectome Dataset Paths ---")
print(f"Neurons CSV:     {CFG.NEURON_CSV}")
print(f"  → Found:       {os.path.exists(CFG.NEURON_CSV)}")
print(f"Connections NPZ: {CFG.CONN_NPZ}")
print(f"  → Found:       {os.path.exists(CFG.CONN_NPZ)}")
print(f"Weights Out:     {CFG.WEIGHTS_PATH}")

## 4. Biological Circuit Architecture
The simulation routes biological signal flow through 6 functional stages:
1. **Sensory Encoding**: Game frames mapped retinotopically to $R1$–$R6$ photoreceptor currents.
2. **Visual Projections**: Lobula visual projection neurons ($LoVP92$, $TmY21$) detect obstacle boundaries.
3. **Central Hubs**: Central brain interneurons integrate inputs with Dale's principle sign constraints.
4. **Descending Commands**: Steering and pursuit descending neurons ($DNg13$, $DNa02$) project to the thoracic ganglion.
5. **Motor Decoding**: Ventral nerve cord wing pre-motor neurons ($TN1A$, $vPR9$, $dPR1$) trigger wing flaps.
6. **Dopaminergic Learning**: $PPL101$ dopamine neurons emit rewards ($+1.0$ pipe, $-5.0$ collision) gating reward-modulated Hebbian plasticity ($R$-STDP) at Kenyon cell synapses.

## 5. High-Speed Synthetic Connectome Simulation
Run 3 fast validation episodes with real-time telemetry and dual-panel dashboard video recording.

In [ ]:
from run import run_simulation

# Run 3 episodes of synthetic connectome with dashboard video recording
results = run_simulation(
    episodes=3,
    mock=True,
    no_viz=False,
    viz_interval=5,
    device=device,
)

print("\n--- Simulation Results ---")
print(f"Episodes Completed: {results['episodes']}")
print(f"Best Score:         {results['best_score']}")
print(f"Video Saved To:     {results['video_path']}")

## 6. Inline Dashboard Video Player
Display the generated simulation video (or animated GIF) directly inside the notebook cell.

In [ ]:
from IPython.display import HTML, Image, display
import base64

video_file = results.get("video_path")
if video_file and os.path.exists(video_file):
    if video_file.endswith(".mp4"):
        with open(video_file, "rb") as f:
            video_b64 = base64.b64encode(f.read()).decode("utf-8")
        html = f"""
        <div style="display: flex; justify-content: center; background: #0d0d0d; padding: 12px; border-radius: 8px;">
            <video width="720" height="420" controls autoplay loop style="border-radius: 6px;">
                <source src="data:video/mp4;base64,{video_b64}" type="video/mp4">
                Your browser does not support HTML5 video.
            </video>
        </div>
        """
        display(HTML(html))
    elif video_file.endswith(".gif"):
        display(Image(filename=video_file, width=720))
else:
    print("No video file found to display.")

## 7. Full-Scale Biological Connectome Run (165,122 Neurons, 6.2M Synapses)
Simulate the real biological brain from the Kaggle dataset (`/kaggle/input/datasets/pernavjain/male-fruit-fly-cns/`).

In [ ]:
if os.path.exists(CFG.CONN_NPZ) and os.path.exists(CFG.NEURON_CSV):
    print(f"Loading full MaleCNS dataset from: {CFG.CONN_NPZ}")
    print(f"Simulating full 165,122-neuron brain on compute device: {device}...")
    results_real = run_simulation(episodes=2, mock=False, no_viz=True, device=device)
    print("\n--- Real Connectome Run Complete ---")
    print(f"Episodes:   {results_real['episodes']}")
    print(f"Best Score: {results_real['best_score']}")
else:
    print("Dataset not found at detected paths. Add the 'male-fruit-fly-cns' dataset to your Kaggle session.")

## References
1. **Berg, S., et al.** (2026). *Sexual dimorphism in the complete Drosophila male central nervous system*. **Cell**, 189, 5504–5526. [doi:10.1016/j.cell.2025.10.045](https://doi.org/10.1016/j.cell.2025.10.045)
2. **Lappalainen, J.K., et al.** (2024). *Connectome-constrained networks predict neural activity across the fly visual system*. **Nature**, 634, 1132–1140. [doi:10.1038/s41586-024-07939-3](https://doi.org/10.1038/s41586-024-07939-3)
3. **Aso, Y., et al.** (2014). *The neuronal architecture of the mushroom body provides a logic for associative learning*. **eLife**, 3:e04577. [doi:10.7554/eLife.04577](https://doi.org/10.7554/eLife.04577)
4. **Dayan, P. & Abbott, L.F.** (2001). *Theoretical Neuroscience*. MIT Press. Ch. 5: Model Neurons I — Leaky Integrate-and-Fire.
5. **Wormuth, A.** (2024). *DoomFly: MaleCNS connectome playing Doom*. [GitHub](https://github.com/awormuth/DoomFly)